# 16 · Autoencoder experiments for healthy versus faulty (Stage R6)

## Purpose
Two questions about the healthy-only autoencoder (AE-6), on development data only (CLAUDE.md §7, Stage R6, declared before
training; seeds 42, 43, 44; the bundle, rules, thresholds and final results are untouched; no reserved recording is read):

1. **Experiment 1:** can more varied healthy-only training views, or a changed objective, improve healthy-vs-faulty
   separation beyond the existing stochastic gain jitter? Arms: **A0** the AE-6 recipe; **A1** fixed recording-level gain
   views with the jitter off; **A2** a denoising objective (jitter on the input, the target un-augmented); **A3** common gain
   + per-channel mismatch + additive noise at the measured floor, with the A2 target.
2. **Experiment 2:** if an autoencoder learns to reconstruct familiar **faults** (**F**), do healthy recordings get higher
   error? With the level controls the brief requires, and an exploratory two-score combination **A0 + F**.

**Correction to the brief, recorded before running.** AE-6's gain jitter is applied to the input and to the target (the
model reconstructs the augmented input), so it adds input diversity but does not teach invariance to level; it is also
already independent per channel. **Predictions recorded before running:** A1 will not differ from A0 (fixed views do what
the continuous jitter already does); A2 is the arm with a mechanism; **F is expected to fail** (a fault recording contains
the healthy machine plus a signature, and Stage R4 showed these autoencoders reconstruct inputs they never trained on).

**The measurement limit, stated before any result.** There are 25 healthy recordings in the development pool, all from
one folder (one session). A healthy false-alarm rate moves in **4-percentage-point steps**; a folder-grouped interval does
not exist on the healthy side (Clopper–Pearson on n = 25 is used and n is stated every time); speed and recording identity
are confounded (Stage R0). A method has to move detection substantially to be visible here at all. **Ceiling:**
augmentation can add robustness to the nuisances it simulates; it cannot create the session and operating diversity that
is missing from 49 healthy recordings of one session.

Training (≈ 50 min, too long for a cell) runs in `scripts/36_r6_train.py`, which only saves per-recording scores; this
notebook defines the protocol, loads those scores and produces every table, figure and results file.

## Inputs
`metadata.csv`, `results/dev_folds.csv`, `results/q1q3/q1_fit_cal_assignment.csv` (the Q1 development folds),
`results/scores/r6_<arm>_s<seed>_dev_scores.parquet` and `results/research/r6_training.json` (from the script), the cache-C
representation (for the augmentation plots), `results/scores/ae6_s42_dev_channel_errors.parquet` (Stage 3B, for the A0
reproduction check). Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the configuration, metadata and the Q1 fold assignment, prints the config and split hashes, and checks the healthy counts against CLAUDE.md §3 (49 normals in total, 25 in the development pool, all from one folder). It then tabulates, per outer fold, the healthy fit and calibration normals (from the other folds), and the fold's own normals and faults that are evaluated.</div>

In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from ml.config import load_config, config_sha256
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import development_pool, folder_proxy, split_hash
from ml.q1 import fold_ids, load_assignment, mild_mask
from ml.rejection import clopper_pearson
from ml.speed_calibration import exceeds, matched_threshold
from ml.specialists import db_to_ln_power, within_class_slope
from ml.ae_experiments import calibrated_z, standardised
from ml.evaluation import mixed_bootstrap_draws

ROOT = SYSTEM_CODE_DIR
R = ROOT / "results"
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
print("config sha256:", config_sha256(cfg)[:16], "| split hash:", split_hash(meta)[:16])
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns).merge(
    pd.read_csv(R / cfg.development.folds_file)[["rec_id", "dev_fold"]], on="rec_id")
normals = pool[pool.class10 == "normal"]
assert (meta.class10 == "normal").sum() == 49 and len(normals) == 25 and len(set(folder_proxy(normals))) == 1, \
    "healthy counts differ from CLAUDE.md §3: stop and ask"
asg = load_assignment(R / "q1q3/q1_fit_cal_assignment.csv")
SEEDS = (42, 43, 44)
roles = []
for k in sorted(asg):
    fit, cal, ev, _ = fold_ids(asg, pool, k)
    evp = pool[pool.rec_id.isin(ev)]
    roles.append({"outer fold": k, "healthy fit (folds ≠ k)": len(fit), "healthy calibration (folds ≠ k)": len(cal),
                  "fault fit for F (folds ≠ k)": int(((pool.dev_fold != k) & (pool.class10 != "normal")).sum()),
                  "evaluated normals (fold k)": int((evp.class10 == "normal").sum()),
                  "evaluated faults (fold k)": int((evp.class10 != "normal").sum())})
ROLES = pd.DataFrame(roles).set_index("outer fold")
print(ROLES.to_string())
print("normals evaluated out of sample over the 3 folds:", ROLES["evaluated normals (fold k)"].sum())

## Split and role diagram
Each outer fold trains on the other folds and evaluates its own recordings once; the healthy fit and calibration normals are
the saved Stage 2D.1 assignment. The fault-only AE (F) trains on the faults of the other folds. Views and augmented copies
exist only inside training and are never calibrated on, evaluated, or counted as recordings.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> draws the roles of every recording group in each outer fold (what trains, what calibrates the threshold, what is evaluated) with the counts from the table above, and saves the diagram for the report.</div>

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
cols = [("healthy fit\n(A0–A3 train)", "#9e9e9e"), ("healthy calibration\n(thresholds)", "#1f77b4"),
        ("fault fit\n(F trains)", "#c9a227"), ("fold k normals\n(evaluated)", "#2ca02c"), ("fold k faults\n(evaluated)", "#d62728")]
keys = list(ROLES.columns)
for i, k in enumerate(ROLES.index):
    for j, ((lab, col), key) in enumerate(zip(cols, keys)):
        ax.add_patch(plt.Rectangle((j * 2.1, -i * 1.1), 1.9, 0.9, color=col, alpha=0.25))
        ax.text(j * 2.1 + 0.95, -i * 1.1 + 0.45, f"{ROLES.loc[k, key]}", ha="center", va="center", fontsize=12)
    ax.text(-0.25, -i * 1.1 + 0.45, f"outer fold {k}", ha="right", va="center", fontsize=10)
for j, (lab, col) in enumerate(cols):
    ax.text(j * 2.1 + 0.95, 1.25, lab, ha="center", va="center", fontsize=9)
ax.set_xlim(-2.2, 10.6); ax.set_ylim(-2.4, 1.8); ax.axis("off")
ax.set_title("Q1 development folds: recordings per role (training views are inside 'fit' and are never counted)", fontsize=10)
fig.tight_layout(); fig.savefig(R / "figures/r6_roles.png", dpi=150); plt.show()

## What the 4-point resolution means
With 25 normals the observable false-alarm rates are 0, 4, 8, 12 … % and their intervals are wide.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints, for 0 to 5 false alarms out of 25 normals, the rate and its 95 % Clopper–Pearson interval: the only operating points this data can distinguish.</div>

In [ ]:
print(pd.DataFrame([{"false alarms": k, "rate": k / 25, "95% CI": "[{:.3f}, {:.3f}]".format(*clopper_pearson(k, 25))}
                    for k in range(6)]).to_string(index=False))

## The augmentation recipes, with a visual check
One frame of one healthy fit recording (fold 0), two channels: the original, A0's jitter (a constant dB offset per channel)
and A3's recipe (a common offset, a per-channel mismatch with σ estimated from the fit normals, and additive noise at the
measured floor F_c, the 5th percentile of each channel's cached dB values over the fit normals). Noise in power can only
raise a value, and raises mainly the quiet bins near the floor.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads one window of one fold-0 healthy fit recording from the representation cache and the A3 noise floor and mismatch SD that the training script measured for fold 0 (seed 42), applies the A0 and A3 augmentations with the same functions the training uses, and plots one frame's spectrum before and after for two channels, plus the change per bin against the original level. The figure is saved.</div>

In [ ]:
import torch
from ml.datasets import load_representation
from ml.spectro import order_axis
from ml.ae_experiments import add_noise_floor, common_mismatch_offsets, jitter_offsets
info = json.loads((R / "research/r6_training.json").read_text())
m3 = next(m for m in info["models"] if m["arm"] == "A3" and m["seed"] == 42 and m["outer_fold"] == 0)
floor, mm = np.asarray(m3["noise_floor_db"]), m3["mismatch_sd_db"]
print("fold 0 noise floor per channel (dB):", np.round(floor, 1), "| per-channel mismatch SD:", round(mm, 3), "dB")
arr, index = load_representation(ROOT / "data" / "representations", cfg, meta)
fit0 = fold_ids(asg, pool, 0)[0]
row = int(index.reset_index(drop=True).index[index.reset_index(drop=True).rec_id == fit0[0]][0])
frame = torch.from_numpy(np.asarray(arr[row][:6, :, 12], dtype=np.float32))[None]          # (1, 6, 128)
g = torch.Generator().manual_seed(0)
a0 = frame + jitter_offsets(g, 1, 6, cfg.ae.gain_jitter_db)[:, :, None]
a3 = add_noise_floor(frame + common_mismatch_offsets(g, 1, 6, cfg.ae.gain_jitter_db, mm)[:, :, None],
                     torch.as_tensor(floor, dtype=torch.float32), g)
orders = order_axis(cfg.preprocess)
names = ["under axial", "under radial", "under tangential", "over axial", "over radial", "over tangential"]
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))
for i, c in enumerate((1, 4)):
    ax = axes[0, i]
    ax.semilogx(orders, frame[0, c], color="k", lw=1.2, label="original")
    ax.semilogx(orders, a0[0, c], color="#4f6bff", lw=1, label="A0 jitter")
    ax.semilogx(orders, a3[0, c], color="#ff8c00", lw=1, label="A3 recipe")
    ax.axhline(floor[c], color="grey", ls=":", lw=1, label="measured floor F_c")
    ax.set_title(f"{names[c]} · {fit0[0]}, one frame", fontsize=9); ax.set_xlabel("order (multiples of shaft speed)")
    ax.set_ylabel("level (dB)"); ax.legend(fontsize=7)
    ax = axes[1, i]
    ax.scatter(frame[0, c], a3[0, c] - frame[0, c], s=6, color="#ff8c00", label="A3 − original")
    ax.scatter(frame[0, c], a0[0, c] - frame[0, c], s=6, color="#4f6bff", label="A0 − original")
    ax.set_xlabel("original level (dB)"); ax.set_ylabel("change (dB)"); ax.legend(fontsize=7)
    ax.set_title("change per bin vs original level (noise raises only quiet bins)", fontsize=9)
fig.tight_layout(); fig.savefig(R / "figures/r6_augmentations.png", dpi=150); plt.show()
assert (a3 - frame - (a3 - frame).min() >= -1e-4).all()

## Training configuration: original recordings and generated views, counted apart

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> summarises the training record written by the script: per arm, the original fit recordings per fold, the generated views (A1 only), the training units (frames, including view copies), the optimiser steps, the training seconds and the parameter count. Views are not recordings: they appear only in their own column.</div>

In [ ]:
T = pd.DataFrame(info["models"])
cfgtab = T.groupby("arm").agg(original_fit_recordings_per_fold=("n_fit_recordings", "mean"),
                              calibration_normals_per_fold=("n_calibration_normals", "mean"),
                              generated_views_per_fold=("n_views", "mean"), training_units=("training_units", "mean"),
                              optimizer_steps=("optimizer_steps", "mean"), train_seconds_mean=("train_seconds", "mean"),
                              train_seconds_total=("train_seconds", "sum"), parameters=("parameters", "first"), models=("arm", "size"))
print(cfgtab.round(1).to_string())
print("healthy fit / calibration normals per fold: 11 / 5-6 (Q1 development folds; not the 20 / 5 of the shared AE-6)")

## Scores on one scale
Per fold, healthy-AE arms: z = log e − log(max calibration-normal error); F: z = log(min calibration-normal error) − log e
(a low fault-AE error means fault-like). Alarm iff z > 0. **F-lc** is F after the level correction (log e − β L, β the pooled
within-class slope on F's own fit recordings, L the cached level in natural-log power units, as in Stage R5). **A0+F** is the
exploratory combination: each score standardised by its calibration normals, then averaged. Every score is pooled over the
three folds; each of the 25 normals and every fault appears once per seed.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads every saved score file, adds labels, speed, severity and family from the metadata, and computes each variant's score on the common scale for every seed and fold (and the calibration normals' scores, which set each fold's threshold). It checks that every development recording is evaluated exactly once per arm and seed, and compares the retrained A0 at seed 42 with Stage 3B's saved AE-6 scores.</div>

In [ ]:
info_m = meta.set_index("rec_id")
raw = pd.concat([pd.read_parquet(R / f"scores/r6_{a}_s{s}_dev_scores.parquet") for a in ("A0", "A1", "A2", "A3", "F") for s in SEEDS])
VARIANTS = ["A0", "A1", "A2", "A3", "F", "F-lc", "A0+F"]
rows, betas = [], []
for s in SEEDS:
    for k in sorted(asg):
        g = raw[(raw.seed == s) & (raw.outer_fold == k)]
        parts = {}
        for a in ("A0", "A1", "A2", "A3", "F"):
            h = g[g.arm == a]
            cal, test = h[h.role == "cal"].error.to_numpy(), h[h.role == "test"]
            parts[a] = (calibrated_z(test.error.to_numpy(), cal, a != "F"), calibrated_z(cal, cal, a != "F"), test.rec_id.to_numpy())
        hf = g[g.arm == "F"]
        fitf = hf[hf.role == "fit"]
        beta = within_class_slope(np.log(fitf.error), db_to_ln_power(fitf["level"]), info_m.loc[fitf.rec_id].class10)
        betas.append({"seed": s, "outer_fold": k, "beta": beta})
        lc = lambda d: np.exp(np.log(d.error.to_numpy()) - beta * db_to_ln_power(d["level"].to_numpy()))
        calf, testf = hf[hf.role == "cal"], hf[hf.role == "test"]
        parts["F-lc"] = (calibrated_z(lc(testf), lc(calf), False), calibrated_z(lc(calf), lc(calf), False), testf.rec_id.to_numpy())
        za, zf = parts["A0"], parts["F"]
        assert (za[2] == zf[2]).all()
        parts["A0+F"] = ((standardised(za[0], za[1]) + standardised(zf[0], zf[1])) / 2,
                         (standardised(za[1], za[1]) + standardised(zf[1], zf[1])) / 2, za[2])
        for v, (zt, zc, ids) in parts.items():
            rows.append(pd.DataFrame({"variant": v, "seed": s, "outer_fold": k, "rec_id": ids, "z": zt - zc.max(),
                                      "cal_max": zc.max()}))
Z = pd.concat(rows, ignore_index=True)
Z = Z.join(info_m[["class10", "fault_family", "severity", "fr_measured_hz"]], on="rec_id")
Z["fault"] = Z.class10 != "normal"
Z["level"] = Z.rec_id.map(raw.drop_duplicates("rec_id").set_index("rec_id")["level"])
assert (Z.groupby(["variant", "seed"]).rec_id.nunique() == len(pool)).all() and not Z.duplicated(["variant", "seed", "rec_id"]).any()
old = pd.read_parquet(R / "scores/ae6_s42_dev_channel_errors.parquet")
old = old[old.role == "test"].set_index("rec_id").score
new = raw[(raw.arm == "A0") & (raw.seed == 42) & (raw.role == "test")].set_index("rec_id").error.loc[old.index]
print(f"A0 (seed 42) vs Stage 3B AE-6: max |diff| {np.abs(new - old).max():.2e}, Spearman {spearmanr(new, old)[0]:.6f} (MPS is not bit-deterministic)")
print("beta (F level correction) per seed and fold:", pd.DataFrame(betas).beta.round(3).tolist())

## Primary: detection at a matched healthy false-alarm count
Per seed, K = the number of the 25 normals that **A0** flags at its own calibrated thresholds; every variant's pooled threshold
is placed to flag exactly K normals, and its detection is the share of fault recordings flagged. AUROC (pooled z, healthy vs
faulty) beside it. The 4-point resolution applies to the false-alarm side; the detection difference vs A0 has a paired
mixed-unit bootstrap per seed (fault folders within class; recordings, never frames or views). Seed spread is reported
separately.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> for each seed computes A0's false-alarm count K at its calibrated thresholds, then each variant's detection at exactly K false alarms out of 25, its AUROC, and the paired bootstrap interval of its detection difference from A0; then the mean and spread over seeds and the declared reading (helps / does not help / too thin to tell).</div>

In [ ]:
def at_count(z_n, z_f, K):
    t = matched_threshold(z_n, K)
    return t, exceeds(z_f, t)
res, rowsK = {}, []
for s in SEEDS:
    zs = Z[Z.seed == s]
    a0 = zs[zs.variant == "A0"]
    K = int(exceeds(a0[~a0.fault].z.to_numpy(), 0.0).sum())
    base_f = a0[a0.fault].sort_values("rec_id")
    folders = np.asarray(folder_proxy(meta.set_index("rec_id").loc[base_f.rec_id].reset_index()))
    draws = mixed_bootstrap_draws(base_f.class10.to_numpy(), folders, cfg.baselines.n_bootstrap, cfg.baselines.seed)
    _, det0 = at_count(a0[~a0.fault].z.to_numpy(), base_f.z.to_numpy(), K)
    for v in VARIANTS:
        x = zs[zs.variant == v]
        xf = x[x.fault].sort_values("rec_id")
        _, det = at_count(x[~x.fault].z.to_numpy(), xf.z.to_numpy(), K)
        diff = det.astype(float) - det0.astype(float)
        boot = np.array([diff[i].mean() for i in draws])
        rowsK.append({"variant": v, "seed": s, "K_false_alarms_of_25": K, "fa_rate": K / 25, "detection": det.mean(),
                      "delta_vs_A0": diff.mean(), "delta_ci95": [float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))],
                      "auroc": roc_auc_score(x.fault, x.z), "n_faults": int(len(xf)), "n_normals": int((~x.fault).sum())})
M = pd.DataFrame(rowsK)
show = M.assign(ci=M.delta_ci95.map(lambda c: f"[{c[0]:+.3f}, {c[1]:+.3f}]"))
print(show[["variant", "seed", "K_false_alarms_of_25", "detection", "delta_vs_A0", "ci", "auroc"]].round(4).to_string(index=False))
summ = M.groupby("variant").agg(detection_mean=("detection", "mean"), detection_sd=("detection", "std"),
                                delta_mean=("delta_vs_A0", "mean"), delta_min=("delta_vs_A0", "min"),
                                delta_max=("delta_vs_A0", "max"), auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"))
print(summ.loc[VARIANTS].round(4).to_string())

## The mildest setting of each family at the matched point, and the declared reading
Criterion 3 requires that no family's mildest setting gets worse (seed mean).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes, at the same matched point, detection at the mildest setting of each fault family (imbalance 6 g, misalignment 0.5 / 0.51 mm, bearings 0 g added mass) for every variant and seed, then applies the three declared criteria against A0 and prints the reading per variant.</div>

In [ ]:
mild = mild_mask(info_m.loc[pool.rec_id].assign(class10=lambda d: d.class10), cfg.anomaly)
mild_rows = []
for s in SEEDS:
    zs = Z[Z.seed == s]
    a0 = zs[zs.variant == "A0"]
    K = int(exceeds(a0[~a0.fault].z.to_numpy(), 0.0).sum())
    for v in VARIANTS:
        x = zs[zs.variant == v].set_index("rec_id")
        t = matched_threshold(x[~x.fault].z.to_numpy(), K)
        for lab, msk in mild.items():
            ids = msk[msk].index
            mild_rows.append({"variant": v, "seed": s, "setting": lab, "n": len(ids), "detection": float(exceeds(x.loc[ids].z.to_numpy(), t).mean())})
MILD = pd.DataFrame(mild_rows)
mt = MILD.groupby(["setting", "variant"]).detection.mean().unstack()[VARIANTS]
print(mt.round(3).to_string())
READING = {}
for v in VARIANTS[1:]:
    d = M[M.variant == v].set_index("seed").delta_vs_A0
    c1 = bool((d > 0).all()); c2 = bool(d.mean() > 0.04); c3 = bool((mt[v] >= mt["A0"] - 1e-12).all())
    reading = "helps" if (c1 and c2 and c3) else ("does not help" if (d.mean() <= 0 or not (d > 0).any()) else "too thin to tell")
    READING[v] = {"higher_all_seeds": c1, "mean_gain_gt_4pp": c2, "mildest_not_worse": c3, "mean_delta": float(d.mean()), "reading": reading}
    print(f"{v:5s}: {reading:17s} | Δ per seed {d.round(4).to_dict()} | mildest not worse: {c3}")

## Each arm at its own calibrated thresholds
Not the primary comparison (false-alarm counts differ), shown for completeness with Clopper–Pearson on n = 25.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> prints, per variant and seed, the healthy false alarms (k of 25, with the 95 % interval) and the detection rate when each fold uses its own calibrated threshold (alarm iff z > 0).</div>

In [ ]:
cal_rows = []
for (v, s), x in Z.groupby(["variant", "seed"]):
    k = int(exceeds(x[~x.fault].z.to_numpy(), 0.0).sum())
    cal_rows.append({"variant": v, "seed": s, "false_alarms": f"{k}/25", "fa_ci95": "[{:.2f}, {:.2f}]".format(*clopper_pearson(k, 25)),
                     "detection": float(exceeds(x[x.fault].z.to_numpy(), 0.0).mean())})
CAL = pd.DataFrame(cal_rows)
print(CAL.pivot_table(index="variant", columns="seed", values=["false_alarms", "detection"], aggfunc="first").loc[VARIANTS].to_string())

## Distributions by class, speed and severity
Score distributions per class for the reference arm, the denoising arm and the fault-only AE (seed 42), and detection at the
matched point by speed band and by severity.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> draws, for A0, A2, F and F-lc at seed 42, the distribution of the common-scale score per class (normal first; the dashed line is the calibrated threshold), saves the figure, and prints detection at the matched point by speed band (< 20, 20–30, ≥ 30 Hz) and by class and severity for every variant (seed mean).</div>

In [ ]:
order = ["normal", "imbalance", "horizontal_misalignment", "vertical_misalignment", "underhang_ball", "underhang_cage",
         "underhang_outer", "overhang_ball", "overhang_cage", "overhang_outer"]
fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=False)
for ax, v in zip(axes, ("A0", "A2", "F", "F-lc")):
    x = Z[(Z.seed == 42) & (Z.variant == v)]
    ax.boxplot([x[x.class10 == c].z for c in order], vert=False, showfliers=False)
    ax.set_yticks(range(1, 11), [c.replace("_", " ") for c in order] if v == "A0" else [""] * 10, fontsize=8)
    ax.axvline(0, ls="--", color="k", lw=1); ax.set_title(v); ax.set_xlabel("score (log units, 0 = calibrated threshold)")
fig.suptitle("Common-scale scores by class, seed 42 (normal = 8-9 recordings per fold, 25 in total)", fontsize=10)
fig.tight_layout(); fig.savefig(R / "figures/r6_scores_by_class.png", dpi=150); plt.show()
Z["band"] = pd.cut(Z.fr_measured_hz, [0, 20, 30, 99], right=False, labels=["<20 Hz", "20-30 Hz", ">=30 Hz"])
br = []
for s in SEEDS:
    zs = Z[Z.seed == s]
    a0 = zs[zs.variant == "A0"]
    K = int(exceeds(a0[~a0.fault].z.to_numpy(), 0.0).sum())
    for v in VARIANTS:
        x = zs[zs.variant == v].copy()
        x["hit"] = exceeds(x.z.to_numpy(), matched_threshold(x[~x.fault].z.to_numpy(), K))
        br.append(x)
B = pd.concat(br)
print(B[B.fault].groupby(["band", "variant"], observed=True).hit.mean().unstack()[VARIANTS].round(3).to_string())
print(B[~B.fault].groupby(["band", "variant"], observed=True).hit.sum().unstack()[VARIANTS].div(3).round(2).to_string(),
      "\n(normals flagged per band, mean over seeds; normals per band:", B[(~B.fault) & (B.variant == "A0") & (B.seed == 42)].band.value_counts().to_dict(), ")")
print(B[B.fault].groupby(["class10", "severity", "variant"]).hit.mean().unstack()[VARIANTS].round(3).to_string())

## Experiment 2: is the fault-only AE's signal level, not knowledge?
Three controls declared before running: the correlation of the F error with the recording's input level (Stage R4's bank
score gave Spearman ≈ +0.60); whether the separation survives the level correction (F-lc); and healthy versus the **three
quietest fault classes** (lowest median level among the fit recordings) separately from the loud ones. If healthy separates
from the loud classes but not from the quiet ones, the effect is level.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes the Spearman correlation of the fault-AE error with input level (all evaluated recordings and within the normals), identifies the three quietest fault classes from the fit recordings' levels, and reports AUROC healthy vs quiet faults and healthy vs loud faults for A0, F and F-lc (seed mean and range); then plots the F error against level and saves the figure.</div>

In [ ]:
fe = raw[(raw.arm == "F") & (raw.role == "test")].join(info_m[["class10"]], on="rec_id")
rho_all = [spearmanr(fe[fe.seed == s].error, fe[fe.seed == s]["level"])[0] for s in SEEDS]
rho_n = [spearmanr(fe[(fe.seed == s) & (fe.class10 == "normal")].error, fe[(fe.seed == s) & (fe.class10 == "normal")]["level"])[0] for s in SEEDS]
print("Spearman(F error, level): all evaluated", np.round(rho_all, 3), "| normals only", np.round(rho_n, 3))
fitlv = raw[(raw.arm == "F") & (raw.role == "fit")].drop_duplicates("rec_id").join(info_m[["class10"]], on="rec_id")
med = fitlv.groupby("class10")["level"].median().sort_values()
QUIET = list(med.index[:3]); LOUD = [c for c in med.index if c not in QUIET]
nlev = Z[(Z.variant == "A0") & (Z.seed == 42) & (~Z.fault)]["level"].median()
print("median fit level by fault class (dB):", med.round(1).to_dict(), "| normals:", round(nlev, 1)); print("quiet:", QUIET)
qr = []
for v in ("A0", "F", "F-lc", "A0+F"):
    for s in SEEDS:
        x = Z[(Z.variant == v) & (Z.seed == s)]
        for grp, cls in (("quiet", QUIET), ("loud", LOUD)):
            y = x[(~x.fault) | x.class10.isin(cls)]
            qr.append({"variant": v, "seed": s, "group": grp, "auroc": roc_auc_score(y.fault, y.z)})
QR = pd.DataFrame(qr)
print(QR.groupby(["variant", "group"]).auroc.agg(["mean", "min", "max"]).round(3).to_string())
fig, ax = plt.subplots(figsize=(8, 5))
x = fe[fe.seed == 42]
for lab, msk, col in (("normal", x.class10 == "normal", "#2ca02c"), ("quiet faults", x.class10.isin(QUIET), "#1f77b4"),
                      ("loud faults", x.class10.isin(LOUD), "#d62728")):
    ax.scatter(x[msk]["level"], np.log(x[msk].error), s=10, alpha=0.6, color=col, label=lab)
ax.set_xlabel("input level (mean cached dB, vibration channels)"); ax.set_ylabel("log fault-AE error")
ax.set_title("Fault-only AE error vs input level, seed 42 (evaluated recordings)", fontsize=10); ax.legend()
fig.tight_layout(); fig.savefig(R / "figures/r6_fault_ae_level.png", dpi=150); plt.show()

## Detection against the false-alarm count
Detection for K = 0 … 5 false alarms out of 25 (0–20 %), seed mean with the seed range: the whole operating range this data
can resolve.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> computes, for every variant and seed, detection at exactly K = 0…5 false alarms among the 25 normals, and draws the seed mean with the seed range; the figure is saved.</div>

In [ ]:
curve = []
for (v, s), x in Z.groupby(["variant", "seed"]):
    for K in range(6):
        curve.append({"variant": v, "seed": s, "K": K, "detection": float(exceeds(x[x.fault].z.to_numpy(), matched_threshold(x[~x.fault].z.to_numpy(), K)).mean())})
CURVE = pd.DataFrame(curve)
fig, ax = plt.subplots(figsize=(8, 5))
for v, col in zip(VARIANTS, plt.cm.tab10.colors):
    c = CURVE[CURVE.variant == v].groupby("K").detection.agg(["mean", "min", "max"])
    ax.errorbar(c.index * 4 + VARIANTS.index(v) * 0.25, c["mean"], yerr=[c["mean"] - c["min"], c["max"] - c["mean"]], marker="o",
                ms=4, capsize=2, color=col, label=v)
ax.set_xlabel("healthy false-alarm rate (%) — steps of 4 points, n = 25"); ax.set_ylabel("fault detection rate")
ax.set_title("Detection at matched false-alarm counts (seed mean, bars = seed range)", fontsize=10); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(R / "figures/r6_matched_curve.png", dpi=150); plt.show()
print(CURVE.groupby(["variant", "K"]).detection.mean().unstack().loc[VARIANTS].round(3).to_string())

## Failure cases
At the matched point, seed 42: the mildest-setting faults the reference arm misses (and whether any arm catches them), and
the normals that raise false alarms (with their speed: the R0 speed caveat applies).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> lists, at seed 42 and the matched point, the mildest-setting fault recordings that A0 misses with the arms that do catch them, and the normal recordings flagged by each variant with their measured speed.</div>

In [ ]:
b42 = B[B.seed == 42].pivot_table(index="rec_id", columns="variant", values="hit", aggfunc="first")
mild_ids = set().union(*[set(m[m].index) for m in mild.values()])
missed = b42.loc[[r for r in b42.index if r in mild_ids and not b42.at[r, "A0"]]]
print(f"mildest-setting faults missed by A0 at seed 42: {len(missed)} of {len(mild_ids)}")
print(missed.join(info_m[["class10", "severity", "fr_measured_hz"]]).sort_values("class10").head(25).to_string())
nf = b42.loc[[r for r in b42.index if info_m.at[r, "class10"] == "normal"]]
print(nf[nf.any(axis=1)].join(info_m[["fr_measured_hz"]]).sort_values("fr_measured_hz").to_string())

## Cost
Training seconds and parameters from the training record; CPU milliseconds per recording (9 windows × 25 frames) measured
here for one autoencoder; every arm has the same architecture, the combination needs two.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> measures the median CPU time of one frame-level autoencoder on one recording's 225 frames (one thread and the default thread count), and prints it with the training cost per arm.</div>

In [ ]:
import time
from ml.models.ae import build_ae
ae = build_ae("frame", 6, cfg.ae).eval()
f = torch.randn(225, 6, 128)
lat = {}
for th in (1, torch.get_num_threads()):
    torch.set_num_threads(th)
    with torch.no_grad():
        for _ in range(3): ae(f)
        ts = []
        for _ in range(30):
            t0 = time.perf_counter(); ae(f); ts.append((time.perf_counter() - t0) * 1000)
    lat[f"{th}_threads"] = float(np.median(ts))
print("CPU ms per recording, one AE:", {k: round(v, 1) for k, v in lat.items()}, "| A0+F needs two AEs")
print(cfgtab[["train_seconds_mean", "train_seconds_total", "parameters", "models"]].round(1).to_string())

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the results: the JSON summary (protocol facts, matched-point table, readings, mildest settings, calibrated thresholds, level controls, cost) and companion CSVs (matched point per variant and seed, the K = 0…5 curve, and every recording's common-scale score per variant and seed).</div>

In [ ]:
out = {"stage": "R6", "seeds": list(SEEDS), "split_hash": split_hash(meta), "config_sha256": config_sha256(cfg),
       "partitions": "Q1 development folds: 11 healthy fit and 5-6 calibration normals per fold; 25 normals evaluated once",
       "measurement_limit": "25 healthy recordings from one folder: false-alarm rate in 4-point steps; Clopper-Pearson n = 25; "
                            "speed and recording identity confounded",
       "ceiling": "augmentation cannot create the session and operating diversity missing from one healthy session",
       "predictions_recorded_before_running": {"A1": "no difference from A0", "F": "expected to fail"},
       "matched_point": M.to_dict("records"), "summary": summ.reset_index().to_dict("records"), "reading": READING,
       "mildest": mt.reset_index().to_dict("records"), "calibrated": CAL.to_dict("records"),
       "level_controls": {"spearman_F_error_level_all": rho_all, "spearman_F_error_level_normals": rho_n,
                          "quiet_classes": QUIET, "auroc_quiet_vs_loud": QR.to_dict("records"), "beta": betas},
       "training": cfgtab.reset_index().to_dict("records"), "cpu_ms_per_recording_one_ae": lat}
(R / "research/r6_ae_experiments.json").write_text(json.dumps(out, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
M.to_csv(R / "research/r6_matched.csv", index=False)
CURVE.to_csv(R / "research/r6_curve.csv", index=False)
Z.drop(columns=["band"]).to_csv(R / "research/r6_per_recording.csv", index=False)
print("written: r6_ae_experiments.json, r6_matched.csv, r6_curve.csv, r6_per_recording.csv")

## Findings

Exploratory development results; seeds 42 / 43 / 44; Q1 development folds (11 fit / 5–6 calibration normals per fold; 25
normals and 779 faults each evaluated once per seed). **25 healthy recordings from one folder: a false-alarm rate moves in
4-point steps (Clopper–Pearson on n = 25), and speed and recording identity are confounded.**

- **Verdict: no arm improves on AE-6 — every variant reads "does not help" by the declared criteria.** At A0's own
  operating point (K = 2 / 1 / 1 false alarms of 25 for seeds 42 / 43 / 44, i.e. 8 / 4 / 4 %), A0 detects 0.974 / 0.951 /
  0.973 of the faults (AUROC 0.991 ± 0.002). Detection difference vs A0: **A1 −0.014 / −0.009 / −0.010**, **A2 −0.009 /
  +0.015 / −0.006** (mean 0.000), **A3 −0.008 / 0.000 / −0.022**; every interval is within ±0.04 of zero and no arm is
  higher on all three seeds. The retrained A0 reproduces Stage 3B's AE-6 scores (max |difference| 8.9e-16).
- **The recorded predictions held.** A1 (110 fixed views per fold, the same 1,500 optimiser steps) is no better than the
  continuous jitter, as predicted, and slightly lower on every seed. A2, the arm with a mechanism (a denoising target), is
  indistinguishable from A0 overall; its only visible movement is at the weakest setting, horizontal misalignment 0.5 mm
  (seed-mean detection 0.587 vs 0.493), which is not enough to change the pooled result and is within seed spread. A3's
  wider recipe (common gain, per-channel mismatch SD 1.8 dB measured from the fit normals, noise at the measured floor) costs
  slightly. Original recordings stayed 11 per fold in every arm; A1's 110 views per fold were never counted as recordings.
- **The fault-only AE fails, as expected:** detection 0.29–0.31 at the matched point, AUROC 0.55 ± 0.01; it flags 4–7 of 25
  normals at its own calibrated threshold. A0 + F (exploratory) is worse than A0 alone (detection 0.71–0.80, AUROC 0.90).
- **Level controls: the fault-AE's signal is dominated by input level.** Spearman(F error, level) = +0.65 to +0.67 over all
  evaluated recordings and +0.70 to +0.76 within the normals (R4's bank: +0.60). The three quietest fault classes
  (imbalance, vertical and horizontal misalignment, median fit level −41 to −42 dB, normals −40.6 dB) are separated from
  healthy with AUROC 0.81, while the loud classes (bearings) give **0.36 — inverted**: loud faults get high error and look
  healthy-like. So F does carry some information at matched level, but level swamps it everywhere else. The declared level
  correction does not change this: the within-class β fitted on F's own fit recordings is tiny (−0.01 to −0.03) and F-lc
  equals F (AUROC 0.55, quiet 0.81, loud 0.34).
- **Where the misses are.** For every arm the weak spot is mild horizontal misalignment: 0.5 mm detection 0.41–0.59, 1.0 mm
  0.62–0.75; at seed 42 A0 misses 14 of the 220 mildest-setting faults, all horizontal misalignment 0.5 mm, and no arm
  recovers more than two of them. The false alarms are a handful of normals that recur across arms (normal__13.1072,
  normal__27.4432 for the healthy-AE arms).
- **Cost and ceiling.** One AE costs 12–13 ms of CPU per recording (A0 + F needs two); training took 6–11 s per healthy-AE
  model and 276 s per fault-AE model (46 min in all). All 49 healthy recordings come from one session: augmentation can add
  robustness to the nuisances it simulates, but it cannot create the missing session and operating diversity — which is
  where any improvement in healthy-vs-faulty separation would have to come from. Nothing is selected or deployed.


## Outputs written
- `results/research/r6_ae_experiments.json`, `r6_matched.csv`, `r6_curve.csv`, `r6_per_recording.csv` (this notebook);
  `results/research/r6_training.json` (the training script).
- Figures: `results/figures/r6_roles.png`, `r6_augmentations.png`, `r6_scores_by_class.png`, `r6_fault_ae_level.png`,
  `r6_matched_curve.png`.
- Scores `results/scores/r6_<arm>_s<seed>_dev_scores.parquet` and weights `artifacts/r6/` (gitignored) from the script.